# Silver Processing Overview
This notebook reads bronze Delta datasets, standardizes column names, trims text fields, applies basic deduplication logic, and writes cleaned silver Delta tables for analytical modeling.

In [0]:
# Purpose: Configure ADLS access and initialize shared paths and batch metadata for silver-layer processing.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import re

storage_account_name = "silveradlsstorage"
bronze_base = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_base = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
batch_id = "manual-20260602-001"

# -- ADLS Gen2 OAuth (Service Principal) auth --
_scope = "retail-adls-kv-scope"
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(scope=_scope, key="adls-sp-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(scope=_scope, key="adls-sp-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=_scope, key='adls-sp-tenant-id')}/oauth2/token"
)


In [0]:
# Purpose: Define helper functions to normalize source column names into a clean silver naming standard.
def clean_column_name(column_name):
    clean_name = column_name.strip().lower()
    clean_name = re.sub(r"[^a-zA-Z0-9_]", "_", clean_name)
    clean_name = re.sub(r"_+", "_", clean_name)
    return clean_name.strip("_")

def standardize_columns(df):
    for old_column in df.columns:
        df = df.withColumnRenamed(old_column, clean_column_name(old_column))
    return df


In [0]:
# Purpose: Define a helper that trims leading and trailing whitespace from all string columns.
def trim_string_columns(df):
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(column_name, F.trim(F.col(column_name)))
    return df


In [0]:
# Purpose: Define deduplication logic that keeps the latest record for each business key when the key exists.
def deduplicate_by_key(df, primary_key):
    if primary_key not in df.columns:
        print(f"Warning: primary key {primary_key} not found. Skipping deduplication.")
        return df
    window_spec = Window.partitionBy(primary_key).orderBy(F.col("ingestedatutc").desc_nulls_last())
    return (
        df
        .withColumn("row_number", F.row_number().over(window_spec))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )


In [0]:
# Purpose: Build a reusable silver transformation pipeline for one entity from bronze read to silver write.
def process_silver_entity(entity_name, primary_key):
    bronze_path = f"{bronze_base}/retail_delta/{entity_name}"
    silver_path = f"{silver_base}/retail_delta_clean/{entity_name}"
    print(f"Reading Bronze: {bronze_path}")
    bronze_df = spark.read.format("delta").load(bronze_path)
    silver_df = standardize_columns(bronze_df)
    silver_df = trim_string_columns(silver_df)
    silver_df = deduplicate_by_key(silver_df, primary_key)
    silver_df = (
        silver_df
        .withColumn("silverprocessedatutc", F.current_timestamp())
        .withColumn("silverbatchid", F.lit(batch_id))
    )
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_path)
    )
    print(f"Completed Silver: {entity_name}")
    print(f"Rows written: {silver_df.count()}")
    return silver_df



In [0]:
# Purpose: Run the silver transformation for orders and preview the cleaned result.
orders_silver_df = process_silver_entity(
    entity_name="orders",
    primary_key="order_id"
)
display(orders_silver_df)



In [0]:
# Purpose: Compare the original bronze orders dataset with the silver output during validation.
display(spark.read.format("delta").load(f"{bronze_base}/retail_delta/orders"))

In [0]:
# Purpose: Define the list of entities to promote to silver and execute the reusable processor for each one.
silver_entities = [
    {"entity_name": "customers", "primary_key": "customer_id"},
    {"entity_name": "products", "primary_key": "product_id"},
    {"entity_name": "orders", "primary_key": "order_id"},
    {"entity_name": "order_items", "primary_key": "order_item_id"},
    {"entity_name": "payments", "primary_key": "payment_id"},
    {"entity_name": "inventory", "primary_key": "inventory_id"},
    {"entity_name": "payment_methods", "primary_key": "payment_method_id"}
]
for entity in silver_entities:
    process_silver_entity(
        entity_name=entity["entity_name"],
        primary_key=entity["primary_key"]
    )


In [0]:
# Purpose: Inspect the payment methods bronze dataset schema and sample rows before silver cleanup.
payment_methods_df = spark.read.format("delta").load(f"{bronze_base}/retail_delta/payment_methods")
payment_methods_df.printSchema()
display(payment_methods_df)



In [0]:
# Purpose: Validate that each silver Delta table was written successfully by checking final row counts.
for entity in silver_entities:
    path = f"{silver_base}/retail_delta_clean/{entity['entity_name']}"
    count_value = spark.read.format("delta").load(path).count()
    print(entity["entity_name"], count_value)

